# 09. Low-Memory Chunked Streaming
For files larger than your available RAM, you can stream the processing chunk-by-chunk using `.iter_begin()`.

In [1]:
import curaster
import numpy as np
import os

# We use the large L size benchmark data
input_file = "../build/benchmark_data/test_L_8192x8192.tif"
print(f"Processing: {input_file}")

Processing: ../build/benchmark_data/test_L_8192x8192.tif


## Stream and Aggregate
We will stream the data and calculate a global histogram without ever loading the full 8192x8192 image into memory at once.

In [ ]:
%%time
queue = curaster.open(input_file) \
    .algebra("B1") \
    .iter_begin(buf_chunks=4)

global_histogram = np.zeros(256, dtype=np.int64)

chunk_count = 0
while True:
    chunk = queue.next()
    if chunk is None:
        break
        
    chunk_count += 1
    data = chunk["data"]
    y0 = chunk["y_offset"]
    h = chunk["height"]
    
    print(f"Received chunk {chunk_count}: y_offset={y0}, size={data.shape}")
    
    hist, _ = np.histogram(data, bins=256, range=(0, 255))
    global_histogram += hist

print(f"\nFinished processing {chunk_count} chunks.")
print(f"Top 5 most frequent pixel values: {np.argsort(global_histogram)[-5:][::-1]}")

Received chunk 1: y_offset=1024, size=(1024, 8192)
Received chunk 2: y_offset=3072, size=(1024, 8192)
Received chunk 3: y_offset=0, size=(1024, 8192)
Received chunk 4: y_offset=2048, size=(1024, 8192)
Received chunk 5: y_offset=4096, size=(1024, 8192)
Received chunk 6: y_offset=5120, size=(1024, 8192)
Received chunk 7: y_offset=7168, size=(1024, 8192)
Received chunk 8: y_offset=6144, size=(1024, 8192)

Finished processing 8 chunks.
Top 5 most frequent pixel values: [255 254 253 252 251]
CPU times: user 1.7 s, sys: 628 ms, total: 2.33 s
Wall time: 988 ms
